In [ ]:
import marimo as mo
import plotly.express as px
import polars as pl

from agentnet import (
    DATA_DIR,
    REVISION_FILE,
    SUBSET_FILES,
    scan_jsonl,
    step_table,
    task_table,
)

In [ ]:
mo.md(f"""
# AgentNet (Ubuntu subset) exploratory data analysis

**Goal.** Understand the Ubuntu part of AgentNet (5,000 tasks) before deciding how to use
it for training: what each field means, which values are missing, and how trajectories look.

**Input.** `data/raw/agentnet/agentnet_ubuntu_5k.jsonl` and `meta_data_merged.jsonl`,
fetched by `scripts/download_agentnet.py`. Dataset revision:
`{REVISION_FILE.read_text().strip()}`. Screenshots are not downloaded.
The Windows and macOS file is excluded for now.

**Conclusion.** To be written after reviewing the results below.
""")

## Field guide

The dataset card documents only some fields and gives no score scales or missing value
rules. The column "Basis" separates the card's wording from what was measured here.

| Field | Meaning | Basis |
|---|---|---|
| `task_id` | Unique id, a timestamp followed by a UUID | measured: unique across all 5,000 rows |
| `instruction` | Human annotated task description | card |
| `natural_language_task` | More natural rewrite of the task, synthesized by a summarizer | card |
| `actual_task` | Detailed task specification, synthesized by a summarizer | card |
| `domain` | Application or category of the task, 12 values | measured, equals `domains` in the metadata |
| `task_completed` | Whether the trajectory solved the task, judged automatically | card says "quality metric"; judge inferred from `reason` |
| `alignment_score` | Judge score for how well the steps match the task, integers 0 to 10 | range measured |
| `efficiency_score` | Judge score for how efficient the steps are, integers 1 to 10 | range measured |
| `task_difficulty` | Judge score for difficulty, integers 0 to 10 | range measured |
| `reason` | Free text justification for the scores | inferred from content, not on the card |
| `traj` | List of steps | card |
| `traj.index` | Zero based step position | measured |
| `traj.image` | Screenshot filename taken before the action | card example |
| `traj.step_id` | Step identifier | not documented |
| `traj.marks` | List of strings attached to a step | not documented, mostly missing |
| `value.observation` | Generated description of the visual scene | card |
| `value.thought` | Reasoning and planning text | card |
| `value.action` | Natural language description of the action | card |
| `value.code` | Executable `pyautogui` or `computer` call; x and y are fractions of the screen in [0, 1] | card for code, range measured |
| `value.last_step_correct` | Whether the previous step was correct | card says "current step"; index 0 is mostly labeled, so semantics unclear |
| `value.last_step_redundant` | Whether the previous step was redundant | card, same caveat |
| `value.reflection` | Error analysis produced by a reflector model | card, mostly missing |

Metadata file (`meta_data_merged.jsonl`): for Ubuntu rows only `task_id`, `instruction`,
`verify_feedback.quality`, `domains`, `system` and `step_num` are filled. Everything else
in that file is only populated for Windows and macOS (checked at the end).

In [ ]:
trajectories = scan_jsonl(DATA_DIR / SUBSET_FILES["ubuntu"])
tasks = (
    task_table(trajectories)
    .with_columns(pl.col("task_completed").is_not_null().alias("scored"))
    .collect()
)

In [ ]:
schema = trajectories.collect_schema()
mo.vstack(
    [
        mo.md("## Schema and missing values per task column"),
        mo.ui.table(
            pl.DataFrame(
                {
                    "column": tasks.columns,
                    "dtype": [str(tasks.schema[column]) for column in tasks.columns],
                    "nulls": [tasks[column].null_count() for column in tasks.columns],
                }
            ).with_columns((pl.col("nulls") / tasks.height).alias("null_rate")),
            selection=None,
        ),
        mo.md(f"Raw file also contains `traj`: `{schema['traj']}`"),
    ]
)

## Unscored tasks

The four quality fields (`task_completed`, `alignment_score`, `efficiency_score`,
`task_difficulty`) and `reason` are null together, and `natural_language_task` and
`actual_task` are null on almost the same rows. Nothing in the card explains this, so these
rows are treated as **unscored**: the quality judge was apparently never run on them. The
share depends strongly on the domain, so scored rows are not a random sample of tasks.

In [ ]:
px.bar(
    tasks.group_by("domain")
    .agg(pl.len().alias("tasks"), (1 - pl.col("scored").mean()).alias("unscored_rate"))
    .sort("unscored_rate"),
    x="unscored_rate",
    y="domain",
    orientation="h",
    color_discrete_sequence=["steelblue"],
    hover_data=["tasks"],
    title="Share of unscored tasks per domain",
)

In [ ]:
check = tasks.select(
    pl.col("scored").sum().alias("scored_rows"),
    (~pl.col("scored")).sum().alias("unscored_rows"),
    (pl.col("alignment_score").is_null() == pl.col("scored")).sum().alias("score_mismatch"),
    (pl.col("reason").is_null() == pl.col("scored")).sum().alias("reason_mismatch"),
    pl.col("natural_language_task").is_null().sum().alias("natural_language_null"),
)
mo.vstack([mo.md("### Null pattern consistency"), mo.ui.table(check, selection=None)])

In [ ]:
overview = tasks.group_by("scored").agg(
    pl.len().alias("tasks"),
    pl.col("task_completed").mean().alias("completed_rate"),
    pl.col("n_steps").mean().alias("mean_steps"),
    pl.col("n_steps").median().alias("median_steps"),
    pl.col("n_steps").max().alias("max_steps"),
)
mo.vstack([mo.md("## Overview"), mo.ui.table(overview, selection=None)])

In [ ]:
px.histogram(
    tasks,
    x="n_steps",
    color="scored",
    nbins=60,
    barmode="overlay",
    opacity=0.6,
    color_discrete_map={True: "steelblue", False: "darkorange"},
    title="Trajectory length (steps)",
)

In [ ]:
px.bar(
    tasks.group_by("domain").agg(pl.len().alias("tasks")).sort("tasks"),
    x="tasks",
    y="domain",
    orientation="h",
    color_discrete_sequence=["steelblue"],
    title="Tasks per domain",
)

In [ ]:
px.histogram(
    tasks.filter(pl.col("scored"))
    .unpivot(
        ["alignment_score", "efficiency_score", "task_difficulty"],
        index="task_id",
        variable_name="score",
    )
    .sort("value"),
    x="value",
    color="score",
    barmode="group",
    color_discrete_map={
        "alignment_score": "steelblue",
        "efficiency_score": "seagreen",
        "task_difficulty": "firebrick",
    },
    title="Judge scores (scored tasks only)",
)

In [ ]:
mo.vstack(
    [
        mo.md("## Scores by completion (scored tasks only)"),
        mo.ui.table(
            tasks.filter(pl.col("scored"))
            .group_by("task_completed")
            .agg(
                pl.len().alias("tasks"),
                pl.col("alignment_score").mean().alias("mean_alignment"),
                pl.col("efficiency_score").mean().alias("mean_efficiency"),
                pl.col("task_difficulty").mean().alias("mean_difficulty"),
                pl.col("n_steps").mean().alias("mean_steps"),
            ),
            selection=None,
        ),
    ]
)

In [ ]:
px.histogram(
    tasks.with_columns(pl.col("instruction").str.len_chars().alias("instruction_chars")),
    x="instruction_chars",
    nbins=60,
    color_discrete_sequence=["steelblue"],
    title="Instruction length (characters)",
)

In [ ]:
steps = (
    step_table(trajectories).collect().join(tasks.select("task_id", "task_completed"), on="task_id")
)

In [ ]:
n = steps.height
missing = pl.DataFrame(
    {
        "field": [
            "last_step_correct",
            "last_step_redundant",
            "reflection (null or empty)",
            "marks",
        ],
        "missing_steps": [
            steps["last_step_correct"].null_count(),
            steps["last_step_redundant"].null_count(),
            steps.filter(
                pl.col("reflection_chars").is_null() | (pl.col("reflection_chars") == 0)
            ).height,
            steps["n_marks"].null_count(),
        ],
    }
).with_columns((pl.col("missing_steps") / n).alias("missing_rate"))
mo.vstack(
    [
        mo.md(f"## Step level missing values ({n:,} steps)"),
        mo.ui.table(missing, selection=None),
    ]
)

In [ ]:
mo.vstack(
    [
        mo.md(
            "## Step labels where present\n\n"
            "Rates over labeled steps only. A null label is not treated as false."
        ),
        mo.ui.table(
            steps.select(
                pl.col("last_step_correct").mean().alias("correct_rate"),
                pl.col("last_step_redundant").mean().alias("redundant_rate"),
                pl.col("thought_chars").mean().alias("mean_thought_chars"),
                pl.col("observation_chars").mean().alias("mean_observation_chars"),
            ),
            selection=None,
        ),
    ]
)

In [ ]:
px.bar(
    steps.group_by("action_name").agg(pl.len().alias("steps")).sort("steps"),
    x="steps",
    y="action_name",
    orientation="h",
    color_discrete_sequence=["steelblue"],
    title="Actions",
)

## Termination status versus `task_completed`

The last step is nearly always `computer.terminate(status=...)`, the agent's own claim of
success or failure. `task_completed` comes from the judge, so disagreement shows how
reliable the agent's self report is. Null `task_completed` marks unscored tasks.

In [ ]:
mo.ui.table(
    steps.filter(pl.col("action_name") == "computer.terminate")
    .group_by("terminate_status", "task_completed")
    .agg(pl.len().alias("tasks"))
    .sort("terminate_status", "task_completed"),
    selection=None,
)

In [ ]:
px.density_heatmap(
    steps.drop_nulls(["x", "y"]),
    x="x",
    y="y",
    nbinsx=40,
    nbinsy=40,
    color_continuous_scale="Blues",
    title="Pointer targets (fractions of screen width and height, y downward)",
).update_yaxes(autorange="reversed")

In [ ]:
meta = scan_jsonl(DATA_DIR / SUBSET_FILES["meta"]).filter(pl.col("system") == "Ubuntu").collect()
joined = tasks.join(meta.select("task_id", "step_num", "domains"), on="task_id")
emptiness = pl.DataFrame(
    {
        "meta_column": meta.columns,
        "null_rate": [meta[column].null_count() / meta.height for column in meta.columns],
    }
)
mo.vstack(
    [
        mo.md(
            f"## Metadata for Ubuntu rows\n\n{meta.height} rows. `step_num` equals "
            f"`n_steps` on {(joined['step_num'] == joined['n_steps']).mean():.1%} of tasks "
            f"and `domains` equals `domain` on "
            f"{(joined['domains'] == joined['domain']).mean():.1%}, so the metadata file "
            "adds no information for Ubuntu."
        ),
        mo.ui.table(emptiness, selection=None),
    ]
)